# Stage 6: SHAP and LIME explanations

Thesis: *Explainable AI for Early Diabetes Prediction* (CDC BRFSS 2015). Config v1.0, master seed 42.

Implements Ch3 3.5 (3.5.1-3.5.4) and decisions **D4, D9, D10, P3, S15**, with S8 (UTF-8), S10 (environment guard first) and S12e (bit-reproducibility).

The ten models were frozen in Stage 4 and evaluated in Stage 5. This notebook only **explains** them: no model, threshold or test metric changes, and no explanation may trigger retuning (Ch3 3.5, 3.7.1).

1. Checks the freeze, every pipeline, `test_predictions.csv` (Stage 5) and the Stage 1 SHAP index files.
2. Draws the 30-case local panel (D10) from the stored test predictions and saves it **before** any explanation runs.
3. SHAP (S15b-c): TreeExplainer (DT, RF, XGB, LGBM, CAT; interventional, 500-row background) and LinearExplainer (LR) on the 2,000-row global sample + panel; PermutationExplainer (SVC, GNB, MLP, KNN; 100-row background, max_evals 250) on the 500-row core + panel; a second permutation seed for SVC, GNB and MLP on the core.
4. LIME (D4): 30 cases x 10 models x 3 seeds, 5,000 perturbations.
5. Global importance, cross-model agreement, stability, direction; local SHAP-LIME agreement, LIME fidelity and seed stability; figures (300 dpi) and a report.

**Resumable.** Every task is saved in chunks under `stage6_outputs/_work/`. If the run stops (sleep, shutdown, an error), just Run All again: finished models and chunks are skipped and the results are identical to an uninterrupted run. KNN runs last.

**Run:** confirm S15, set `S15_CONFIRMED = True` in Cell 2, put the notebook in `thesis_xai/`, keep the laptop on mains power with sleep disabled, Run All. Expected time on the PC about 2.6 h (worst case about 4.5 h), most of it KNN SHAP (about 1.8 h) and Random Forest SHAP (about 25 min); Cell 6 prints the plan and Cell 7 an estimated time remaining.

In [1]:
# Cell 1 - environment guard (S10): runs before numpy / scikit-learn are imported.
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/thesis_xai"   # edit if your Drive folder differs
else:
    PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import env_setup
env_setup.ensure_pinned(PROJECT_DIR, install=IN_COLAB)

All pinned packages present at the pinned versions.


{}

In [2]:
# Cell 2 - configuration (S15). The run stops here until S15 is confirmed.
S15_CONFIRMED = True          # set to True only after S15 is confirmed in the decisions log
if not S15_CONFIRMED:
    raise SystemExit("Confirm S15 first, then set S15_CONFIRMED = True and run again.")

import hashlib
import json
import math
import platform
import time
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
from joblib import Parallel, delayed
from scipy.stats import kendalltau, rankdata, spearmanr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import thesis_common as tc

MACHINE = "colab" if IN_COLAB else "pc"
STAGE = "stage6_explanations.ipynb"
FREEZE_SHA256 = "1dd5bf06708d148399675d9fc657857df1c86f015cbd83c24e79e8754b48c0d8"   # Stage 4 freeze (S4-047)
PRED_SHA256 = "b9813461cd03e4552b50c245c8750648d08b9c48c1960d249642982bc15114ed"     # Stage 5 predictions (S5-136)
EXPECTED = {"n_raw": 253680, "test_rows": 50736, "test_pos": 7069, "bg": (500, 70), "perm_bg": (100, 14),
            "global": (2000, 279), "core": (500, 70)}
MODELS = list(tc.MODELS)
FEATURES = list(tc.FEATURES)
BIN_IDX = [i for i, f in enumerate(FEATURES) if f not in tc.NON_BINARY]        # the 14 binary predictors
TREE = ["DT", "RF", "XGB", "LGBM", "CAT"]
PERM = ["SVC", "KNN", "GNB", "MLP"]
SCALE = {m: "probability" for m in MODELS}
SCALE.update({"LR": "log-odds", "XGB": "log-odds", "LGBM": "log-odds", "CAT": "log-odds"})
MAX_EVALS = 250                 # P3
LIME_SAMPLES = 5000             # D4
TOP_K = 5                       # D4: top-5 Jaccard
PANEL_PER_CAT = 5               # D10
CATEGORIES = ["high_probability", "borderline", "TP", "FN", "FP", "TN"]   # draw order (S15f)
POOL_SHARE = 0.01               # S15f: top / lowest 1% pools
TREE_WORKERS = max(1, min(6, (os.cpu_count() or 2) // 2))   # PC: 6 physical cores (S15i)
TREE_CHUNK, PERM_CHUNK = 50, 10
MAX_ROWS_PER_TASK = {}          # testing only; must stay {} for the thesis run


def after_chunk(task, k):       # testing hook (no-op)
    pass


# Seeds (S2 + S15b): a 14th stream 'shap_permutation' is appended; the first 13 must be unchanged.
SS14 = np.random.SeedSequence(tc.MASTER_SEED).generate_state(14)
assert [int(v) for v in SS14[:13]] == [tc.SEEDS[k] for k in tc.SEED_STREAMS], "seed derivation changed"
SEED_SHAP_PERM = int(SS14[13])
PERM_SEEDS = [int(s) for s in np.random.SeedSequence(SEED_SHAP_PERM).generate_state(2)]     # seed 1, seed 2
LIME_SEEDS = [int(s) for s in np.random.SeedSequence(tc.SEEDS["lime_perturbation"]).generate_state(3)]
SEED_CASES = tc.SEEDS["lime_case_selection"]


def row_seed(base, row_id):
    # one seed per explained row: results do not depend on chunking, order or interruptions
    return int(np.random.SeedSequence([int(base), int(row_id)]).generate_state(1)[0])


OUT = os.path.join(PROJECT_DIR, "stage6_outputs")
WORK, FIG = os.path.join(OUT, "_work"), os.path.join(OUT, "figures")
for d in [OUT, WORK, FIG, os.path.join(OUT, "shap"), os.path.join(OUT, "lime")]:
    os.makedirs(d, exist_ok=True)
TEST_MODE = bool(MAX_ROWS_PER_TASK)
print(f"machine={MACHINE}  python={platform.python_version()}  shap={shap.__version__}  tree workers={TREE_WORKERS}")
print(f"seeds: shap_permutation={SEED_SHAP_PERM} -> {PERM_SEEDS}; lime={LIME_SEEDS}; cases={SEED_CASES}")

C:\Users\USER\Downloads\thesis_xai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


machine=pc  python=3.13.0  shap=0.52.0  tree workers=6
seeds: shap_permutation=3957024168 -> [2941821818, 2285058082]; lime=[2792219722, 1652670301, 1080463383]; cases=2468990893


In [3]:
# Cell 3 - freeze, pipelines, Stage 5 predictions and Stage 1 SHAP index files.
def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()


fz_path = os.path.join(PROJECT_DIR, "stage4_outputs", "frozen_models.json")
if tc.sha256_file(fz_path) != FREEZE_SHA256:
    raise SystemExit("frozen_models.json is not the Stage 4 freeze. Restore it.")
with open(fz_path, encoding="utf-8") as f:
    FZ = json.load(f)
for m in MODELS:
    r = FZ["models"][m]
    if tc.sha256_file(os.path.join(PROJECT_DIR, r["pipeline"])) != r["pipeline_sha256"]:
        raise SystemExit(f"{m}: {r['pipeline']} differs from the freeze. Restore it.")
PIPE = {m: os.path.join(PROJECT_DIR, FZ["models"][m]["pipeline"]) for m in MODELS}
T_MCC = {m: FZ["models"][m]["threshold_mcc"] for m in MODELS}

pred_path = os.path.join(PROJECT_DIR, "stage5_outputs", "test_predictions.csv")
if not os.path.exists(pred_path) or tc.sha256_file(pred_path) != PRED_SHA256:
    raise SystemExit("stage5_outputs/test_predictions.csv is missing or is not the Stage 5 file. Restore it.")
PRED = pd.read_csv(pred_path, float_precision="round_trip")

X_all, y_all = tc.load_data(PROJECT_DIR)               # SHA-256 checked
idx = tc.load_indices(PROJECT_DIR)                      # every index file checked against MANIFEST.csv
ROW_ID_NAMES = ["row_id", "row", "idx", "index"]
dev_df = idx["dev_folds.csv"]
dev_rows = dev_df[next(c for c in ROW_ID_NAMES if c in dev_df.columns)].to_numpy()
test_rows = np.setdiff1d(np.arange(len(X_all)), dev_rows)
DEV_SET, TEST_SET = set(dev_rows.tolist()), set(test_rows.tolist())


def int_cols(df):
    return [c for c in df.columns if pd.api.types.is_integer_dtype(df[c])]


def sample_ids(name, universe, n_exp):
    # the one integer column of unique raw-row ids inside the right partition (matched by content, not name)
    df = idx[name]
    c = [c for c in int_cols(df) if df[c].is_unique and len(df) == n_exp and set(df[c].tolist()) <= universe]
    if len(c) != 1:
        raise SystemExit(f"{name}: cannot identify the row-id column (candidates {c}; columns {list(df.columns)})")
    return np.sort(df[c[0]].to_numpy())


def resamples(name, base, n_rep=5):
    # long file: one column labels the resample (n_rep values x len(base) rows); one column holds raw row ids
    # or positions in `base`, with repeats (a bootstrap) -> list of n_rep arrays of positions in `base`
    df = idx[name]
    if len(df) != n_rep * len(base):
        raise SystemExit(f"{name}: {len(df)} rows, expected {n_rep * len(base)}")
    lab = [c for c in int_cols(df) + [c for c in df.columns if df[c].dtype == object]
           if df[c].nunique() == n_rep and (df[c].value_counts() == len(base)).all()]
    if len(lab) != 1:
        raise SystemExit(f"{name}: cannot identify the resample column (candidates {lab})")
    # Raw row ids are preferred; positions (0 .. n-1) are used only if no column holds row ids.
    # Columns with at most two distinct values (e.g. the class label y) are never row columns.
    bset, ids, poss = set(base.tolist()), {}, {}
    for c in [c for c in int_cols(df) if c != lab[0] and df[c].nunique() > 2]:
        v = df[c].to_numpy()
        if set(v.tolist()) <= bset:
            pos, store = np.searchsorted(base, v), ids
        elif v.min() >= 0 and v.max() < len(base):
            pos, store = v, poss
        else:
            continue
        groups = [pos[(df[lab[0]] == g).to_numpy()] for g in sorted(df[lab[0]].unique())]
        if all(len(np.unique(g)) < len(g) for g in groups):       # bootstrap draws repeat rows
            store[c] = groups
    out = ids or poss
    if len(out) != 1:
        raise SystemExit(f"{name}: cannot identify the resampled-row column (candidates {list(out)}; columns {list(df.columns)})")
    return next(iter(out.values()))


BG = sample_ids("shap_background_500_dev.csv", DEV_SET, EXPECTED["bg"][0])
BG100 = sample_ids("shap_perm_background_100_dev.csv", DEV_SET, EXPECTED["perm_bg"][0])
GLOBAL = sample_ids("shap_global_sample_2000_test.csv", TEST_SET, EXPECTED["global"][0])
CORE = sample_ids("shap_perm_core_500_test.csv", TEST_SET, EXPECTED["core"][0])
STAB = {"global": resamples("shap_stability_bootstrap_global_2000.csv", GLOBAL),
        "core": resamples("shap_stability_bootstrap_core_500.csv", CORE)}
y_full = y_all
checks = {
    "test rows": (len(test_rows), EXPECTED["test_rows"]), "test positives": (int(y_full[test_rows].sum()), EXPECTED["test_pos"]),
    "test_predictions rows = test partition": (np.array_equal(PRED.row_id.to_numpy(), test_rows), True),
    "test_predictions labels = data": (np.array_equal(PRED.y.to_numpy(), y_full[test_rows]), True),
    "background 500: positives": (int(y_full[BG].sum()), EXPECTED["bg"][1]),
    "permutation background 100: positives": (int(y_full[BG100].sum()), EXPECTED["perm_bg"][1]),
    "permutation background is a subset of the 500 background": (set(BG100) <= set(BG), True),
    "global sample 2000: positives": (int(y_full[GLOBAL].sum()), EXPECTED["global"][1]),
    "core 500: positives": (int(y_full[CORE].sum()), EXPECTED["core"][1]),
    "core is a subset of the global sample": (set(CORE) <= set(GLOBAL), True),
    "global stability: 5 within-class bootstraps": (all(int(y_full[GLOBAL[g]].sum()) == EXPECTED["global"][1] for g in STAB["global"]), True),
    "core stability: 5 within-class bootstraps": (all(int(y_full[CORE[g]].sum()) == EXPECTED["core"][1] for g in STAB["core"]), True),
}
for k, (got, want) in checks.items():
    print(f"{'PASS' if got == want else 'FAIL'}  {k}: {got}")
if any(got != want for got, want in checks.values()):
    raise SystemExit("Input checks failed.")

X_dev = X_all.iloc[dev_rows].reset_index(drop=True)
XBG, XBG100 = X_all.iloc[BG].reset_index(drop=True), X_all.iloc[BG100].reset_index(drop=True)

# The frozen pipelines must reproduce the stored Stage 5 probabilities exactly. This is checked on the full test set,
# because KNN's choice among neighbours tied at the 251st distance depends on the batch size in scikit-learn
# (a 2,000-row batch changes a few probabilities by 1/251); the full set is the batch Stage 5 used.
prow = pd.Series(np.arange(len(PRED)), index=PRED.row_id.to_numpy())
repro = {}
for m in MODELS:
    pipe = joblib.load(PIPE[m])
    if m == "RF":
        pipe.set_params(clf__n_jobs=1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        p = pipe.predict_proba(X_all.iloc[test_rows])[:, 1]
    d = float(np.max(np.abs(p - PRED[f"p_{m}"].to_numpy())))
    repro[m] = d
    if d != 0.0:
        raise SystemExit(f"{m}: frozen pipeline does not reproduce the Stage 5 probabilities (max diff {d:.3g}).")
del pipe
print("Stage 5 test probabilities reproduced exactly for all ten models (max abs diff):",
      {m: f"{v:.1e}" for m, v in repro.items()})

PASS  test rows: 50736
PASS  test positives: 7069
PASS  test_predictions rows = test partition: True
PASS  test_predictions labels = data: True
PASS  background 500: positives: 70
PASS  permutation background 100: positives: 14
PASS  permutation background is a subset of the 500 background: True
PASS  global sample 2000: positives: 279
PASS  core 500: positives: 70
PASS  core is a subset of the global sample: True
PASS  global stability: 5 within-class bootstraps: True
PASS  core stability: 5 within-class bootstraps: True
Stage 5 test probabilities reproduced exactly for all ten models (max abs diff): {'LR': '0.0e+00', 'DT': '0.0e+00', 'RF': '0.0e+00', 'SVC': '0.0e+00', 'KNN': '0.0e+00', 'GNB': '0.0e+00', 'XGB': '0.0e+00', 'LGBM': '0.0e+00', 'CAT': '0.0e+00', 'MLP': '0.0e+00'}


In [4]:
# Cell 4 - local case panel (D10, S15f), drawn from the stored test predictions and saved before any explanation.
Pm = PRED[[f"p_{m}" for m in MODELS]].to_numpy()
Tm = np.array([T_MCC[m] for m in MODELS])
yt, rid = PRED.y.to_numpy(), PRED.row_id.to_numpy()
n_t = len(PRED)
vote = (Pm >= Tm).sum(axis=1)                                               # models voting positive at their MCC threshold
pct = np.column_stack([rankdata(Pm[:, j], method="average") / n_t for j in range(len(MODELS))])
med_pct = np.median(pct, axis=1)                                            # model-neutral risk percentile
t_pct = (Pm < Tm).mean(axis=0)                                              # percentile position of each threshold
border = np.median(np.abs(pct - t_pct), axis=1)                             # model-neutral distance to the thresholds
k_pool = math.ceil(POOL_SHARE * n_t)
POOLS = {"high_probability": rid[np.lexsort((rid, -med_pct))[:k_pool]],
         "borderline": rid[np.lexsort((rid, border))[:k_pool]],
         "TP": rid[(yt == 1) & (vote >= 6)], "FN": rid[(yt == 1) & (vote <= 4)],
         "FP": rid[(yt == 0) & (vote >= 6)], "TN": rid[(yt == 0) & (vote <= 4)]}
rng = np.random.default_rng(SEED_CASES)
taken, rows = set(), []
for cat in CATEGORIES:
    cand = np.array(sorted(set(POOLS[cat].tolist()) - taken))
    pick = rng.choice(cand, PANEL_PER_CAT, replace=False)
    for r_ in pick:
        i = int(prow.loc[r_])
        rows.append({"case_id": len(rows) + 1, "category": cat, "row_id": int(r_), "y": int(yt[i]), "vote_positive": int(vote[i]),
                     "median_percentile": med_pct[i], "median_threshold_distance": border[i], "pool_size": len(cand),
                     **{f"p_{m}": Pm[i, j] for j, m in enumerate(MODELS)}})
    taken |= set(int(v) for v in pick)
panel = pd.DataFrame(rows)
blob = panel.to_csv(index=False, float_format="%.17g", lineterminator="\n").encode("utf-8")
panel_path = os.path.join(OUT, "case_panel.csv")
if os.path.exists(panel_path):
    with open(panel_path, "rb") as f:
        if f.read() != blob:
            raise SystemExit("case_panel.csv exists and differs from this draw. It is never overwritten; report this.")
    print("case_panel.csv: verified identical to the saved panel")
else:
    with open(panel_path, "wb") as f:
        f.write(blob)
    print("case_panel.csv: created")
PANEL = panel.row_id.to_numpy()
print(panel[["case_id", "category", "row_id", "y", "vote_positive", "median_percentile"]].round(3).to_string(index=False))
print("pool sizes:", {c: len(v) for c, v in POOLS.items()}, "| 5-5 votes excluded from TP/TN/FP/FN:", int((vote == 5).sum()))

case_panel.csv: created
 case_id         category  row_id  y  vote_positive  median_percentile
       1 high_probability  145108  1             10              0.989
       2 high_probability  157007  1             10              0.994
       3 high_probability   69262  1             10              0.991
       4 high_probability  185865  1             10              0.992
       5 high_probability  134354  1             10              0.995
       6       borderline  149348  1              2              0.747
       7       borderline   67812  0              4              0.758
       8       borderline   97328  0              6              0.773
       9       borderline   42809  0              5              0.767
      10       borderline   39027  0              9              0.775
      11               TP    9209  1             10              0.811
      12               TP  163874  1             10              0.975
      13               TP  139637  1              8  

In [5]:
# Cell 5 - explainers (S15b). Workers are self-contained so they can run in separate processes.
def model_output(m, pipe, X):
    # the output each explainer decomposes (D9: each model's native scale)
    clf = pipe.named_steps["clf"] if hasattr(pipe, "named_steps") else None
    if m in ("DT", "RF"):
        return clf.predict_proba(X)[:, 1]
    if m == "XGB":
        return clf.predict(X, output_margin=True).astype(float)
    if m == "LGBM":
        return clf.predict(X, raw_score=True)
    if m == "CAT":
        return clf.predict(X, prediction_type="RawFormulaVal")
    if m == "LR":
        return pipe.decision_function(X)
    return pipe.predict_proba(X)[:, 1]


def tree_chunk(path, m, Xc, Xbg, k):
    import time, warnings
    import joblib, numpy as np, shap
    t0, c0 = time.perf_counter(), time.process_time()
    pipe = joblib.load(path)
    clf = pipe.named_steps["clf"]
    if m == "RF":
        clf.set_params(n_jobs=1)
    elif m in ("XGB", "LGBM"):
        clf.set_params(n_jobs=1)                            # CatBoost cannot change parameters after fitting
    obj = clf.get_booster() if m == "XGB" else clf          # shap 0.52 + xgboost 3.4.1: the sklearn wrapper fails (S15b)
    import contextlib, io
    with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):   # no progress bars from workers
        warnings.simplefilter("ignore")
        ex = shap.TreeExplainer(obj, shap.maskers.Independent(Xbg, max_samples=len(Xbg)),   # all 500 rows (S15b)
                                feature_perturbation="interventional", model_output="raw")
        e = ex(Xc, check_additivity=False)
        fx = model_output(m, pipe, Xc)
    v, b = np.asarray(e.values), np.asarray(e.base_values, dtype=float)
    if v.ndim == 3:                                         # DT / RF: per-class output, keep the positive class (S6)
        v, b = v[:, :, 1], b[:, 1]
    b = np.broadcast_to(b, (len(Xc),)).astype(float)
    return k, v.astype(float), b, np.asarray(fx, float), time.process_time() - c0, time.perf_counter() - t0


def linear_all(path, Xc, Xbg):
    t0 = time.perf_counter()
    pipe = joblib.load(path)
    prep, clf = pipe.named_steps["prep"], pipe.named_steps["clf"]
    names = list(prep.get_feature_names_out())               # the scaled columns come first (S15b)
    Zbg, Z = prep.transform(Xbg), prep.transform(Xc)
    e = shap.LinearExplainer(clf, shap.maskers.Independent(Zbg, max_samples=len(Zbg)))(Z)
    order = [names.index(f) for f in FEATURES]
    v = np.asarray(e.values)[:, order]
    b = np.broadcast_to(np.asarray(e.base_values, float), (len(Xc),)).astype(float)
    return v, b, model_output("LR", pipe, Xc), time.perf_counter() - t0


class Dedup:
    # exact cache of repeated rows within one call (the model is deterministic, so values are unchanged)
    def __init__(self, pipe):
        self.pipe = pipe

    def __call__(self, Z):
        U, inv = np.unique(Z, axis=0, return_inverse=True)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=DeprecationWarning)
            p = self.pipe.predict_proba(pd.DataFrame(U, columns=FEATURES))[:, 1]
        return p[inv.ravel()]


def perm_explainer(m):
    pipe = joblib.load(PIPE[m])
    f = Dedup(pipe)
    ex = shap.PermutationExplainer(f, shap.maskers.Independent(XBG100.to_numpy(), max_samples=len(XBG100)))
    return pipe, f, ex


def perm_rows(ex, f, Xc, ids, seed):
    vals, base = [], []
    for x, r_ in zip(Xc, ids):
        np.random.seed(row_seed(seed, r_))                  # shap draws permutations from numpy's global generator
        e = ex(x[None, :], max_evals=MAX_EVALS, silent=True)
        vals.append(e.values[0]); base.append(float(np.asarray(e.base_values).ravel()[0]))
    return np.array(vals, float), np.array(base), f(Xc)


def lime_explainer():
    from lime.lime_tabular import LimeTabularExplainer
    return LimeTabularExplainer(X_dev.to_numpy(), feature_names=FEATURES, categorical_features=BIN_IDX,
                                discretize_continuous=True, discretizer="quartile", mode="classification",
                                class_names=["0", "1"], random_state=0)


def lime_case(LE, predict_fn, x, seed):
    rs = np.random.RandomState(seed)                       # equal to a fresh explainer built with this seed
    LE.random_state = rs
    LE.discretizer.random_state = rs
    e = LE.explain_instance(x, predict_fn, labels=(1,), num_features=len(FEATURES), num_samples=LIME_SAMPLES)
    pairs = e.as_map()[1]
    w = dict(pairs)
    labels = {j: s for (j, _), (s, _) in zip(pairs, e.as_list(label=1))}     # as_list follows the as_map order
    return w, labels, float(e.score), float(np.ravel(e.local_pred)[0]), float(e.intercept[1])

In [6]:
# Cell 6 - task list, checkpoints and the time plan (S15i). Cheapest first; KNN last.
SHAP_ROWS = {"tree": np.union1d(GLOBAL, PANEL), "perm": np.union1d(CORE, PANEL), "seed2": CORE}
TASKS = ([("SHAP", m, 1) for m in ["GNB", "LR", "DT", "SVC", "MLP", "LGBM", "CAT", "XGB", "RF"]]
         + [("SHAP", m, 2) for m in ["GNB", "SVC", "MLP"]]
         + [("LIME", m, 1) for m in ["GNB", "LR", "DT", "SVC", "MLP", "LGBM", "CAT", "XGB", "RF"]]
         + [("SHAP", "KNN", 1), ("LIME", "KNN", 1)])
# planning rates for the PC, seconds (full-size kit test 2026-10-14, scaled to the PC); used only for the time estimate
PLAN_TREE_CORE_S = {"DT": 0.003, "LGBM": 0.20, "CAT": 0.26, "XGB": 0.31, "RF": 4.2}     # per row, one core
PLAN_PERM_S = {"GNB": 0.06, "SVC": 0.14, "MLP": 0.07, "KNN": 12.5}                       # per row
PLAN_LIME_S = {"KNN": 5.7, "RF": 0.3, "LGBM": 0.2}                                                    # per explanation (others 0.05)


def task_name(kind, m, s):
    return f"{kind}_{m}" + ("_seed2" if s == 2 else "")


def task_rows(kind, m, s):
    if kind == "LIME":
        rows = PANEL
    elif s == 2:
        rows = SHAP_ROWS["seed2"]
    else:
        rows = SHAP_ROWS["perm"] if m in PERM else SHAP_ROWS["tree"]
    lim = MAX_ROWS_PER_TASK.get(m)
    if lim and kind == "SHAP":                                  # testing only: panel rows first, then the core
        rows = np.r_[np.intersect1d(rows, PANEL), np.setdiff1d(rows, PANEL)][:lim]
    return np.asarray(rows)


def planned_seconds(kind, m, s):
    n_ = len(task_rows(kind, m, s))
    if kind == "LIME":
        return n_ * len(LIME_SEEDS) * PLAN_LIME_S.get(m, 0.05) + 20
    if m == "LR":
        return 5
    if m in TREE:
        return n_ * PLAN_TREE_CORE_S[m] / (0.8 * TREE_WORKERS) + 10
    return n_ * PLAN_PERM_S[m] + 5


def done_path(kind, m, s):
    return os.path.join(OUT, kind.lower(), task_name(kind, m, s) + ".done.json")


def read_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def write_atomic(path, data):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(data)
    os.replace(tmp, path)


def is_done(kind, m, s):
    p = done_path(kind, m, s)
    if not os.path.exists(p):
        return False
    d = read_json(p)
    if tc.sha256_file(os.path.join(OUT, d["file"])) != d["sha256"]:
        raise SystemExit(f"{d['file']} differs from its .done record. Delete both and rerun this task.")
    return True


def task_manifest(kind, m, s, chunk):
    wd = os.path.join(WORK, task_name(kind, m, s))
    os.makedirs(wd, exist_ok=True)
    rows = task_rows(kind, m, s)
    man = {"rows_sha256": sha_bytes(rows.astype(np.int64).tobytes()), "n_rows": len(rows), "chunk": chunk,
           "pipeline_sha256": FZ["models"][m]["pipeline_sha256"], "max_evals": MAX_EVALS,
           "perm_seed": PERM_SEEDS[s - 1], "lime_seeds": LIME_SEEDS, "shap": shap.__version__}
    mp = os.path.join(wd, "manifest.json")
    if os.path.exists(mp):
        if read_json(mp) != man:
            raise SystemExit(f"{mp} was written with other settings. Delete the folder {wd} and rerun.")
    else:
        write_atomic(mp, json.dumps(man, indent=1).encode("utf-8"))
    return wd, rows


plan = pd.DataFrame([{"task": task_name(*t), "rows": len(task_rows(*t)) * (len(LIME_SEEDS) if t[0] == "LIME" else 1),
                      "planned_min": planned_seconds(*t) / 60, "status": "done" if is_done(*t) else "to run"} for t in TASKS])
print(plan.round(1).to_string(index=False))
todo_min = plan.loc[plan.status == "to run", "planned_min"].sum()
print(f"\nPlanned time for the remaining tasks: {todo_min:.0f} min (about {todo_min / 60:.1f} h); worst case about 1.8x.")

          task  rows  planned_min status
      SHAP_GNB   530          0.6 to run
       SHAP_LR  2029          0.1 to run
       SHAP_DT  2029          0.2 to run
      SHAP_SVC   530          1.3 to run
      SHAP_MLP   530          0.7 to run
     SHAP_LGBM  2029          1.6 to run
      SHAP_CAT  2029          2.0 to run
      SHAP_XGB  2029          2.4 to run
       SHAP_RF  2029         29.8 to run
SHAP_GNB_seed2   500          0.6 to run
SHAP_SVC_seed2   500          1.2 to run
SHAP_MLP_seed2   500          0.7 to run
      LIME_GNB    90          0.4 to run
       LIME_LR    90          0.4 to run
       LIME_DT    90          0.4 to run
      LIME_SVC    90          0.4 to run
      LIME_MLP    90          0.4 to run
     LIME_LGBM    90          0.6 to run
      LIME_CAT    90          0.4 to run
      LIME_XGB    90          0.4 to run
       LIME_RF    90          0.8 to run
      SHAP_KNN   530        110.5 to run
      LIME_KNN    90          8.9 to run

Planned time fo

In [7]:
# Cell 7 - run every task (resumable). Progress and estimated time remaining are printed as chunks finish.
SPEED = []                       # measured / planned time of tasks finished in this session
LAST_PRINT = [0.0]


def progress(msg, final=False):
    # at most one progress line per 30 s (and always the last one)
    now = time.perf_counter()
    if final or now - LAST_PRINT[0] >= 30:
        print(msg, flush=True)
        LAST_PRINT[0] = now


def eta_text(cur_left_s, after):
    f = float(np.median(SPEED)) if SPEED else 1.0
    rest = sum(planned_seconds(*t) for t in after if not is_done(*t)) * f
    return f"task ETA {cur_left_s / 60:.1f} min; all remaining about {(cur_left_s + rest) / 60:.0f} min"


def finish_shap(kind, m, s, rows, V, Bv, FX, wall, cpu, workers, n_chunks):
    err = np.abs(Bv + V.sum(axis=1) - FX)
    df = pd.DataFrame(V, columns=FEATURES)
    df.insert(0, "additivity_error", err)
    df.insert(0, "model_output", FX)
    df.insert(0, "base_value", Bv)
    df.insert(0, "row_id", rows)
    rel = os.path.join("shap", task_name(kind, m, s) + ".csv")
    blob = df.to_csv(index=False, float_format="%.17g", lineterminator="\n").encode("utf-8")
    write_atomic(os.path.join(OUT, rel), blob)
    rec = {"file": rel, "sha256": sha_bytes(blob), "rows": len(rows), "chunks": n_chunks, "wall_s": wall, "worker_cpu_s": cpu,
           "workers": workers, "explainer": {"tree": "TreeExplainer (interventional, 500 bg)", "linear": "LinearExplainer (500 bg)",
           "perm": f"PermutationExplainer (100 bg, max_evals {MAX_EVALS})"}[("tree" if m in TREE else "linear" if m == "LR" else "perm")],
           "output_scale": SCALE[m], "max_additivity_error": float(err.max()), "machine": MACHINE,
           "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    write_atomic(done_path(kind, m, s), json.dumps(rec, indent=1).encode("utf-8"))
    return rec


def load_chunks(wd, n_chunks):
    got = {}
    for k in range(n_chunks):
        p = os.path.join(wd, f"c{k:04d}.npz")
        if os.path.exists(p):
            with np.load(p) as z:
                got[k] = {key: z[key] for key in z.files}
    return got


def save_chunk(wd, k, **arrs):
    p = os.path.join(wd, f"c{k:04d}.npz")
    tmp = p + ".tmp.npz"
    np.savez(tmp, **arrs)
    os.replace(tmp, p)


def run_shap(t, after):
    kind, m, s = t
    chunk = 10 ** 9 if m == "LR" else TREE_CHUNK if m in TREE else PERM_CHUNK
    wd, rows = task_manifest(kind, m, s, chunk)
    Xr = X_all.iloc[rows].reset_index(drop=True)
    starts = list(range(0, len(rows), chunk))
    got = load_chunks(wd, len(starts))
    todo = [k for k in range(len(starts)) if k not in got]
    t0 = time.perf_counter()
    print(f"[{task_name(*t)}] {len(rows)} rows, {len(starts)} chunks, {len(got)} already saved", flush=True)
    rows_done = sum(len(got[k]["v"]) for k in got)
    if m == "LR":
        if todo:
            v, b, fx, w = linear_all(PIPE[m], Xr, XBG)
            save_chunk(wd, 0, v=v, b=b, fx=fx, wall=np.array([w]), cpu=np.array([w]))
    elif m in TREE:
        jobs = (delayed(tree_chunk)(PIPE[m], m, Xr.iloc[starts[k]:starts[k] + chunk], XBG, k) for k in todo)
        n_left = sum(len(Xr.iloc[starts[k]:starts[k] + chunk]) for k in todo)
        done_here = 0
        for k, v, b, fx, cpu, w in Parallel(n_jobs=TREE_WORKERS, return_as="generator_unordered")(jobs):
            save_chunk(wd, k, v=v, b=b, fx=fx, wall=np.array([w]), cpu=np.array([cpu]))
            rows_done += len(v); n_left -= len(v); done_here += len(v)
            el = time.perf_counter() - t0
            progress(f"  {rows_done}/{len(rows)} rows, {el / 60:.1f} min; " + eta_text(n_left * el / done_here, after), n_left == 0)
            after_chunk(task_name(*t), k)
    else:
        pipe, f, ex = perm_explainer(m)
        Xn = Xr.to_numpy()
        n_left = sum(len(Xn[starts[k]:starts[k] + chunk]) for k in todo)
        done_here = 0
        for k in todo:
            sl = slice(starts[k], starts[k] + chunk)
            c0, w0 = time.process_time(), time.perf_counter()
            v, b, fx = perm_rows(ex, f, Xn[sl], rows[sl], PERM_SEEDS[s - 1])
            w = time.perf_counter() - w0
            save_chunk(wd, k, v=v, b=b, fx=fx, wall=np.array([w]), cpu=np.array([time.process_time() - c0]))
            rows_done += len(v); n_left -= len(v); done_here += len(v)
            el = time.perf_counter() - t0
            progress(f"  {rows_done}/{len(rows)} rows, {el / 60:.1f} min; " + eta_text(n_left * el / done_here, after), n_left == 0)
            after_chunk(task_name(*t), k)
    got = load_chunks(wd, len(starts))
    V = np.vstack([got[k]["v"] for k in range(len(starts))])
    Bv = np.concatenate([got[k]["b"] for k in range(len(starts))])
    FX = np.concatenate([got[k]["fx"] for k in range(len(starts))])
    cpu = float(sum(got[k]["cpu"][0] for k in got))
    if m not in TREE:
        wall = float(sum(got[k]["wall"][0] for k in got))       # sequential chunks: the sum is the task's wall time
    elif todo == list(range(len(starts))):
        wall = time.perf_counter() - t0                          # parallel chunks, one session: elapsed time
    else:                                                        # resumed: estimate from the chunk times
        wall = float(sum(got[k]["wall"][0] for k in got)) / TREE_WORKERS
    return finish_shap(kind, m, s, rows, V, Bv, FX, wall, cpu, TREE_WORKERS if m in TREE else 1, len(starts))


LE = None


def run_lime(t, after):
    global LE
    kind, m, s = t
    wd, rows = task_manifest(kind, m, s, 1)
    if LE is None:
        LE = lime_explainer()
    pipe = joblib.load(PIPE[m])
    if m == "RF":
        pipe.set_params(clf__n_jobs=1)

    def predict_fn(Z):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=DeprecationWarning)
            return pipe.predict_proba(pd.DataFrame(Z, columns=FEATURES))

    W, S = [], []
    t0, n_all = time.perf_counter(), len(rows) * len(LIME_SEEDS)
    for i, r_ in enumerate(rows):
        x = X_all.iloc[int(r_)].to_numpy().astype(float)
        p_model = float(predict_fn(x[None, :])[0, 1])
        case = int(panel.case_id[panel.row_id == r_].iloc[0])
        for si, seed in enumerate(LIME_SEEDS):
            w, labels, r2, lp, icpt = lime_case(LE, predict_fn, x, seed)
            for j in range(len(FEATURES)):
                W.append({"case_id": case, "row_id": int(r_), "model": m, "seed_index": si + 1, "feature": FEATURES[j],
                          "condition": labels.get(j, ""), "weight": w.get(j, 0.0)})
            S.append({"case_id": case, "row_id": int(r_), "model": m, "seed_index": si + 1, "seed": seed, "r2": r2,
                      "local_pred": lp, "intercept": icpt, "model_p": p_model, "abs_gap": abs(lp - p_model)})
        k_done = (i + 1) * len(LIME_SEEDS)
        el = time.perf_counter() - t0
        progress(f"  [{task_name(*t)}] {k_done}/{n_all} explanations, {el / 60:.1f} min; "
                 + eta_text((n_all - k_done) * el / k_done, after), i + 1 == len(rows))
    wall = time.perf_counter() - t0
    W, S = pd.DataFrame(W), pd.DataFrame(S)
    rel = os.path.join("lime", task_name(*t) + ".csv")
    blob = W.to_csv(index=False, float_format="%.17g", lineterminator="\n").encode("utf-8")
    write_atomic(os.path.join(OUT, rel), blob)
    S.to_csv(os.path.join(OUT, "lime", task_name(*t) + "_fidelity.csv"), index=False, float_format="%.17g", lineterminator="\n")
    rec = {"file": rel, "sha256": sha_bytes(blob), "rows": len(rows), "explanations": n_all, "wall_s": wall, "workers": 1,
           "machine": MACHINE, "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    write_atomic(done_path(*t), json.dumps(rec, indent=1).encode("utf-8"))
    return rec


t_session = time.perf_counter()
for i, t in enumerate(TASKS):
    if is_done(*t):
        continue
    t1 = time.perf_counter()
    rec = run_shap(t, TASKS[i + 1:]) if t[0] == "SHAP" else run_lime(t, TASKS[i + 1:])
    used = time.perf_counter() - t1
    SPEED.append(used / planned_seconds(*t))
    extra = f"max additivity error {rec['max_additivity_error']:.2e}" if t[0] == "SHAP" else ""
    print(f"[{task_name(*t)}] done in {used / 60:.1f} min {extra}", flush=True)
print(f"All {len(TASKS)} tasks finished. This session: {(time.perf_counter() - t_session) / 60:.1f} min.")

[SHAP_GNB] 530 rows, 53 chunks, 0 already saved
  10/530 rows, 0.1 min; task ETA 6.2 min; all remaining about 170 min
  530/530 rows, 0.5 min; task ETA 0.0 min; all remaining about 164 min
[SHAP_GNB] done in 0.5 min max additivity error 2.55e-15
[SHAP_LR] 2029 rows, 1 chunks, 0 already saved
[SHAP_LR] done in 0.0 min max additivity error 2.22e-15
[SHAP_DT] 2029 rows, 41 chunks, 0 already saved
  2029/2029 rows, 0.1 min; task ETA 0.0 min; all remaining about 68 min
[SHAP_DT] done in 0.1 min max additivity error 6.42e-08
[SHAP_SVC] 530 rows, 53 chunks, 0 already saved
  270/530 rows, 0.5 min; task ETA 0.5 min; all remaining about 99 min
  530/530 rows, 1.0 min; task ETA 0.0 min; all remaining about 98 min
[SHAP_SVC] done in 1.0 min max additivity error 1.78e-15
[SHAP_MLP] 530 rows, 53 chunks, 0 already saved
  530/530 rows, 0.5 min; task ETA 0.0 min; all remaining about 108 min
[SHAP_MLP] done in 0.5 min max additivity error 1.33e-15
[SHAP_LGBM] 2029 rows, 41 chunks, 0 already saved
  35

In [8]:
# Cell 8 - load results; global importance, cross-model agreement, stability, permutation-seed check, direction.
def load_shap(m, s=1):
    d = pd.read_csv(os.path.join(OUT, "shap", task_name("SHAP", m, s) + ".csv"), float_precision="round_trip")
    return d.set_index("row_id")


SV = {m: load_shap(m) for m in MODELS}
SV2 = {m: load_shap(m, 2) for m in ["GNB", "SVC", "MLP"]}
CORE_USED = np.array(sorted(set(CORE).intersection(*[set(SV[m].index) for m in MODELS])))
if not TEST_MODE and len(CORE_USED) != len(CORE):
    raise SystemExit("A model's SHAP file does not cover the 500-row core.")
OWN = {m: (CORE if m in PERM else GLOBAL) for m in MODELS}


def importance(A):
    a = np.abs(A).mean(axis=0)
    return a, a / a.sum()


def top_k(v, k=TOP_K):
    return set(np.argsort(-v, kind="stable")[:k].tolist())


def jacc(a, b):
    return len(a & b) / len(a | b)


IMP_CORE, rows = {}, []
for m in MODELS:
    for sample, ids in [("core_500", CORE_USED), ("own", OWN[m] if not TEST_MODE else np.array(sorted(set(OWN[m]) & set(SV[m].index))))]:
        raw, nrm = importance(SV[m].loc[ids, FEATURES].to_numpy())
        if sample == "core_500":
            IMP_CORE[m] = nrm
        rk = rankdata(-nrm, method="average")
        rows += [{"model": m, "sample": sample, "n_rows": len(ids), "output_scale": SCALE[m], "feature": f,
                  "mean_abs_shap": raw[j], "normalized_importance": nrm[j], "rank": rk[j]} for j, f in enumerate(FEATURES)]
GIMP = pd.DataFrame(rows)
GIMP.to_csv(os.path.join(OUT, "global_importance.csv"), index=False, float_format="%.10g")

pairs = []
for i, a in enumerate(MODELS):
    for b_ in MODELS[i + 1:]:
        tau = kendalltau(IMP_CORE[a], IMP_CORE[b_]).statistic             # tau-b (ties handled)
        pairs.append({"model_a": a, "model_b": b_, "kendall_tau_b": tau,
                      "top5_jaccard": jacc(top_k(IMP_CORE[a]), top_k(IMP_CORE[b_]))})
GPAIRS = pd.DataFrame(pairs)
GPAIRS.to_csv(os.path.join(OUT, "global_agreement_pairs.csv"), index=False, float_format="%.10g")

st_rows = []
for m in MODELS:
    for sample, base, groups in [("own", OWN[m], STAB["core" if m in PERM else "global"]), ("core_500", CORE, STAB["core"])]:
        if TEST_MODE and not set(base) <= set(SV[m].index):
            continue
        A = SV[m].loc[base, FEATURES].to_numpy()
        _, full = importance(A)
        for g, pos in enumerate(groups):
            _, r_ = importance(A[pos])
            st_rows.append({"model": m, "sample": sample, "resample": g + 1, "kendall_tau_b": kendalltau(full, r_).statistic,
                            "top5_jaccard": jacc(top_k(full), top_k(r_))})
GSTAB = pd.DataFrame(st_rows, columns=["model", "sample", "resample", "kendall_tau_b", "top5_jaccard"])
GSTAB.to_csv(os.path.join(OUT, "global_stability.csv"), index=False, float_format="%.10g")

sc_rows = []
for m in SV2:
    ids = np.array(sorted(set(CORE) & set(SV2[m].index) & set(SV[m].index)))
    A1, A2 = SV[m].loc[ids, FEATURES].to_numpy(), SV2[m].loc[ids, FEATURES].to_numpy()
    _, i1 = importance(A1); _, i2 = importance(A2)
    sc_rows.append({"model": m, "n_rows": len(ids), "kendall_tau_b_global": kendalltau(i1, i2).statistic,
                    "top5_jaccard_global": jacc(top_k(i1), top_k(i2)),
                    "mean_abs_diff_over_mean_abs_shap": np.abs(A1 - A2).mean() / np.abs(A1).mean(),
                    "pearson_all_values": float(np.corrcoef(A1.ravel(), A2.ravel())[0, 1])})
SEEDCHK = pd.DataFrame(sc_rows)
SEEDCHK.to_csv(os.path.join(OUT, "perm_seed_check.csv"), index=False, float_format="%.10g")

dr = []
Xc = X_all.iloc[CORE_USED]
for m in MODELS:
    for f in FEATURES:
        x_, s_ = Xc[f].to_numpy(), SV[m].loc[CORE_USED, f].to_numpy()
        rho = spearmanr(x_, s_).statistic if np.ptp(x_) > 0 and np.ptp(s_) > 0 else np.nan
        dr.append({"model": m, "feature": f, "spearman_value_vs_shap": rho})
DIRN = pd.DataFrame(dr)
DIRN.to_csv(os.path.join(OUT, "shap_direction.csv"), index=False, float_format="%.10g")

fid = []
for m in MODELS:
    for s, d in [(1, SV[m])] + ([(2, SV2[m])] if m in SV2 else []):
        rec = read_json(done_path("SHAP", m, s))
        fid.append({"model": m, "seed": s, "explainer": rec["explainer"], "output_scale": SCALE[m], "n_rows": len(d),
                    "max_abs_additivity_error": d.additivity_error.max(), "mean_abs_additivity_error": d.additivity_error.mean()})
SFID = pd.DataFrame(fid)
SFID.to_csv(os.path.join(OUT, "shap_fidelity.csv"), index=False, float_format="%.10g")

pd.set_option("display.width", 220)
print("Normalized global importance (core, %):")
print((pd.DataFrame(IMP_CORE, index=FEATURES) * 100).round(1).to_string())
print("\nRank agreement over 45 model pairs: tau-b median", round(GPAIRS.kendall_tau_b.median(), 3),
      "range", round(GPAIRS.kendall_tau_b.min(), 3), "-", round(GPAIRS.kendall_tau_b.max(), 3),
      "| top-5 Jaccard median", round(GPAIRS.top5_jaccard.median(), 3))
print("\nStability (5 bootstrap resamples):")
print(GSTAB.groupby(["model", "sample"])[["kendall_tau_b", "top5_jaccard"]].agg(["mean", "min"]).round(3).to_string())
print("\nPermutation seed check:\n" + SEEDCHK.round(4).to_string(index=False))
print("\nSHAP additivity:\n" + SFID.to_string(index=False))

Normalized global importance (core, %):
                        LR    DT    RF   SVC   KNN   GNB   XGB  LGBM   CAT   MLP
HighBP                16.1  15.8  15.9  14.0  14.9   8.7  13.0  13.3  12.7  13.0
HighChol              12.3  10.9  11.8  11.7  11.2   5.8  10.7  10.7  10.4  10.6
CholCheck              3.4   0.3   0.8   1.5   0.3   2.1   3.1   2.8   3.2   1.5
BMI                   11.7  16.0  16.4  14.8  13.3   5.6  13.7  13.8  13.8  15.6
Smoker                 0.3   0.5   0.6   0.7   1.3   1.7   0.4   0.5   0.5   1.6
Stroke                 0.4   0.0   0.8   1.1   0.4   4.0   0.6   0.6   0.6   0.6
HeartDiseaseorAttack   1.4   0.9   3.4   4.1   2.2   7.1   1.9   1.9   2.0   2.6
PhysActivity           0.7   0.6   1.1   0.9   1.5   4.1   0.6   0.6   0.5   1.3
Fruits                 1.3   0.5   0.8   1.1   1.3   1.2   0.8   0.8   0.9   1.0
Veggies                0.3   0.4   0.7   0.3   0.8   2.3   0.4   0.4   0.3   0.9
HvyAlcoholConsump      3.2   1.2   1.2   2.0   0.7   2.4   2.2   2.0 

In [9]:
# Cell 9 - local: SHAP-LIME agreement, LIME fidelity and seed stability, cross-model local agreement.
LW = pd.concat([pd.read_csv(os.path.join(OUT, "lime", f"LIME_{m}.csv"), float_precision="round_trip") for m in MODELS])
LF = pd.concat([pd.read_csv(os.path.join(OUT, "lime", f"LIME_{m}_fidelity.csv"), float_precision="round_trip") for m in MODELS])
LF.to_csv(os.path.join(OUT, "lime_fidelity.csv"), index=False, float_format="%.10g")
cat_of = dict(zip(panel.case_id, panel.category))


def lime_vec(m, case, si=None):
    d = LW[(LW.model == m) & (LW.case_id == case)]
    if si is not None:
        d = d[d.seed_index == si]
        return d.set_index("feature").weight.reindex(FEATURES).to_numpy()
    return d.groupby("feature").weight.mean().reindex(FEATURES).to_numpy()


agr, seedst, cross = [], [], []
for _, c in panel.iterrows():
    phis = {}
    for m in MODELS:
        if c.row_id not in SV[m].index:
            continue
        phi = SV[m].loc[c.row_id, FEATURES].to_numpy()
        phis[m] = phi
        w = lime_vec(m, c.case_id)
        t5 = np.argsort(-np.abs(phi), kind="stable")[:TOP_K]
        f_ = LF[(LF.model == m) & (LF.case_id == c.case_id)]
        agr.append({"case_id": c.case_id, "category": c.category, "model": m,
                    "kendall_tau_b_abs": kendalltau(np.abs(phi), np.abs(w)).statistic,
                    "top5_jaccard": jacc(top_k(np.abs(phi)), top_k(np.abs(w))),
                    "sign_agreement_shap_top5": float(np.mean(np.sign(phi[t5]) == np.sign(w[t5]))),
                    "lime_r2_mean": f_.r2.mean(), "lime_abs_gap_mean": f_.abs_gap.mean()})
        vs = [lime_vec(m, c.case_id, si) for si in range(1, len(LIME_SEEDS) + 1)]
        for a in range(len(vs)):
            for b_ in range(a + 1, len(vs)):
                seedst.append({"case_id": c.case_id, "model": m, "seeds": f"{a + 1}-{b_ + 1}",
                               "kendall_tau_b_abs": kendalltau(np.abs(vs[a]), np.abs(vs[b_])).statistic,
                               "top5_jaccard": jacc(top_k(np.abs(vs[a])), top_k(np.abs(vs[b_])))})
    ms = list(phis)
    for i, a in enumerate(ms):
        for b_ in ms[i + 1:]:
            cross.append({"case_id": c.case_id, "category": c.category, "model_a": a, "model_b": b_,
                          "kendall_tau_b_abs": kendalltau(np.abs(phis[a]), np.abs(phis[b_])).statistic,
                          "top5_jaccard": jacc(top_k(np.abs(phis[a])), top_k(np.abs(phis[b_])))})
AGR, SEEDST, CROSS = pd.DataFrame(agr), pd.DataFrame(seedst), pd.DataFrame(cross)
AGR.to_csv(os.path.join(OUT, "shap_lime_agreement.csv"), index=False, float_format="%.10g")
SEEDST.to_csv(os.path.join(OUT, "lime_seed_stability.csv"), index=False, float_format="%.10g")
CROSS.to_csv(os.path.join(OUT, "local_crossmodel_agreement.csv"), index=False, float_format="%.10g")

LSUM = AGR.groupby("model")[["kendall_tau_b_abs", "top5_jaccard", "sign_agreement_shap_top5", "lime_r2_mean",
                            "lime_abs_gap_mean"]].median().join(
    SEEDST.groupby("model")[["kendall_tau_b_abs", "top5_jaccard"]].median().add_prefix("lime_seed_")).reindex(MODELS)
LSUM.to_csv(os.path.join(OUT, "local_summary_by_model.csv"), float_format="%.10g")
print("Local summary (medians over the 30 cases):\n" + LSUM.round(3).to_string())
print("\nCross-model local agreement, median tau-b by category:\n"
      + CROSS.groupby("category").kendall_tau_b_abs.median().reindex(CATEGORIES).round(3).to_string())

Local summary (medians over the 30 cases):
       kendall_tau_b_abs  top5_jaccard  sign_agreement_shap_top5  lime_r2_mean  lime_abs_gap_mean  lime_seed_kendall_tau_b_abs  lime_seed_top5_jaccard
model                                                                                                                                                 
LR                 0.571         0.429                       1.0         0.537              0.037                        0.800                   1.000
DT                 0.510         1.000                       1.0         0.407              0.057                        0.700                   1.000
RF                 0.505         0.667                       1.0         0.545              0.052                        0.843                   1.000
SVC                0.429         0.429                       1.0         0.503              0.035                        0.805                   0.667
KNN                0.538         0.667             

In [10]:
# Cell 10 - timing (operational, S15i) and figures (300 dpi).
tim = []
for t in TASKS:
    rec = read_json(done_path(*t))
    n_units = rec.get("explanations", rec["rows"])
    tim.append({"task": task_name(*t), "kind": t[0], "model": t[1], "units": n_units, "workers": rec["workers"],
                "wall_s": rec["wall_s"], "worker_cpu_s": rec.get("worker_cpu_s", np.nan),
                "wall_s_per_unit": rec["wall_s"] / n_units,
                "single_core_s_per_row": rec["worker_cpu_s"] / n_units if t[1] in TREE and t[0] == "SHAP" else np.nan,
                "machine": rec["machine"]})
TIM = pd.DataFrame(tim)
TIM.to_csv(os.path.join(OUT, "timing_stage6.csv"), index=False, float_format="%.6g")
print(TIM.round(3).to_string(index=False))

plt.rcParams.update({"font.size": 8.5, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "figure.facecolor": "white", "axes.facecolor": "white"})
SEQ = LinearSegmentedColormap.from_list("seq", ["#f7f6f3", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
VAL = LinearSegmentedColormap.from_list("val", ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
C_SHAP, C_LIME = "#2a78d6", "#eb6834"

# (1) normalized importance heatmap (core, all ten models); features ordered by mean rank
IMPDF = pd.DataFrame(IMP_CORE, index=FEATURES)
order_f = IMPDF.rank(ascending=False).mean(axis=1).sort_values().index
H = IMPDF.loc[order_f, MODELS]
fig, ax = plt.subplots(figsize=(6.8, 6.6))
ax.imshow(H.to_numpy(), cmap=SEQ, aspect="auto", vmin=0, vmax=float(H.to_numpy().max()))
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.iat[i, j]
        ax.text(j, i, f"{100 * v:.0f}" if v >= 0.005 else "<1", ha="center", va="center", fontsize=6.5,
                color="white" if v > 0.55 * H.to_numpy().max() else "#0b0b0b")
ax.set_xticks(range(len(MODELS)), MODELS); ax.set_yticks(range(len(order_f)), order_f)
ax.tick_params(length=0); [s.set_visible(False) for s in ax.spines.values()]
ax.set_title(f"Normalized mean |SHAP| (% of each model's total), common core ({len(CORE_USED)} test rows)", fontsize=8.5)
fig.savefig(os.path.join(FIG, "fig_shap_importance_heatmap.png"), dpi=300, bbox_inches="tight"); plt.close(fig)

# (2) cross-model rank agreement (Kendall tau-b)
TM = pd.DataFrame(np.eye(len(MODELS)), index=MODELS, columns=MODELS)
for r_ in GPAIRS.itertuples():
    TM.loc[r_.model_a, r_.model_b] = TM.loc[r_.model_b, r_.model_a] = r_.kendall_tau_b
fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(TM.to_numpy(), cmap=DIV, vmin=-1, vmax=1)
for i in range(len(MODELS)):
    for j in range(len(MODELS)):
        ax.text(j, i, f"{TM.iat[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if abs(TM.iat[i, j]) > 0.75 else "#0b0b0b")
ax.set_xticks(range(len(MODELS)), MODELS); ax.set_yticks(range(len(MODELS)), MODELS)
ax.tick_params(length=0); [s.set_visible(False) for s in ax.spines.values()]
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.outline.set_visible(False); cb.set_label("Kendall tau-b")
ax.set_title("Agreement of global feature rankings (21 predictors)", fontsize=8.5)
fig.savefig(os.path.join(FIG, "fig_shap_rank_agreement.png"), dpi=300, bbox_inches="tight"); plt.close(fig)

# (3) beeswarm-style plots for the primary contrast (CAT and LR, 2,000-row global sample)
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.8))
jit = np.random.default_rng(0)                                   # jitter for display only
for ax, m in zip(axes, [FZ["contrasts"]["primary"][0], FZ["contrasts"]["primary"][1]]):
    ids = np.array(sorted(set(GLOBAL) & set(SV[m].index)))
    A = SV[m].loc[ids, FEATURES]
    feats = A.abs().mean().sort_values(ascending=False).index[:12][::-1]
    Xg = X_all.iloc[ids]
    for i, f in enumerate(feats):
        xv = Xg[f].to_numpy()
        col = rankdata(xv) / len(xv)
        ax.scatter(A[f].to_numpy(), i + jit.uniform(-0.3, 0.3, len(ids)), c=col, cmap=VAL, s=3, lw=0, alpha=0.7, rasterized=True)
    ax.axvline(0, color="#52514e", lw=0.8)
    ax.set_yticks(range(len(feats)), feats); ax.tick_params(axis="y", length=0)
    ax.set_xlabel(f"SHAP value ({SCALE[m]})"); ax.set_title(f"{tc.MODEL_NAMES[m]} ({len(ids)} test rows)", fontsize=8.5)
    ax.grid(axis="x", color="#e6e5e0", lw=0.6)
sm_ = plt.cm.ScalarMappable(cmap=VAL); sm_.set_array([0, 1])
cb = fig.colorbar(sm_, ax=axes, fraction=0.025, pad=0.02, ticks=[0, 1]); cb.ax.set_yticklabels(["low", "high"])
cb.set_label("Feature value (percentile)"); cb.outline.set_visible(False)
fig.savefig(os.path.join(FIG, "fig_shap_beeswarm_primary_contrast.png"), dpi=300, bbox_inches="tight"); plt.close(fig)

# (4) SHAP vs LIME on the first-drawn TP and FN cases (fixed rule), primary-contrast models
cases = [int(panel[panel.category == "TP"].case_id.iloc[0]), int(panel[panel.category == "FN"].case_id.iloc[0])]
fig, axes = plt.subplots(2, 2, figsize=(9.6, 6.4))
for r_, case in enumerate(cases):
    row_id = int(panel[panel.case_id == case].row_id.iloc[0])
    for c_, m in enumerate(FZ["contrasts"]["primary"]):
        ax = axes[r_, c_]
        phi = SV[m].loc[row_id, FEATURES].to_numpy(); w = lime_vec(m, case)
        top = np.argsort(-np.abs(phi), kind="stable")[:8][::-1]
        ps, ws = phi / np.abs(phi).sum(), w / np.abs(w).sum()
        yy = np.arange(len(top))
        ax.barh(yy + 0.2, ps[top], height=0.38, color=C_SHAP, label="SHAP")
        ax.barh(yy - 0.2, ws[top], height=0.38, color=C_LIME, label="LIME (mean of 3 seeds)")
        ax.axvline(0, color="#52514e", lw=0.8)
        ax.set_yticks(yy, [FEATURES[j] for j in top]); ax.tick_params(axis="y", length=0)
        p_ = panel[panel.case_id == case].iloc[0][f"p_{m}"]
        ax.set_title(f"Case {case} ({cat_of[case]}, y = {int(panel[panel.case_id == case].y.iloc[0])}) - {m}, p = {p_:.3f}", fontsize=8.5)
        ax.grid(axis="x", color="#e6e5e0", lw=0.6)
axes[1, 0].set_xlabel("Share of total |attribution| (signed)"); axes[1, 1].set_xlabel("Share of total |attribution| (signed)")
axes[0, 1].legend(frameon=False, fontsize=7.5, loc="lower right")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_shap_vs_lime_cases.png"), dpi=300); plt.close(fig)

# (5) SHAP-LIME agreement per model over the 30 cases
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
ax = axes[0]                                                      # continuous: box plot with the 30 cases as dots
data = [AGR[AGR.model == m]["kendall_tau_b_abs"].dropna().to_numpy() for m in MODELS]
ax.boxplot(data, widths=0.5, showfliers=False, medianprops={"color": "#0b0b0b", "lw": 1.2},
           boxprops={"color": "#52514e"}, whiskerprops={"color": "#52514e"}, capprops={"color": "#52514e"})
for j, d in enumerate(data):
    ax.scatter(j + 1 + jit.uniform(-0.15, 0.15, len(d)), d, s=8, color=C_SHAP, lw=0, alpha=0.75, zorder=3)
ax.set_xticks(range(1, len(MODELS) + 1), MODELS); ax.set_ylabel("Kendall tau-b, |SHAP| vs |LIME|")
ax.grid(axis="y", color="#e6e5e0", lw=0.6)
ax = axes[1]                                                      # discrete values (0, 0.25, 0.43, 0.67, 1): mean as a bar
mj = AGR.groupby("model").top5_jaccard.mean().reindex(MODELS)
ax.bar(range(len(MODELS)), mj.to_numpy(), width=0.6, color=C_SHAP)
for j, v in enumerate(mj.to_numpy()):
    ax.text(j, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=7, color="#0b0b0b")
ax.set_xticks(range(len(MODELS)), MODELS); ax.set_ylim(0, 1.1); ax.set_ylabel("Mean top-5 Jaccard, SHAP vs LIME")
ax.grid(axis="y", color="#e6e5e0", lw=0.6)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_shap_lime_agreement.png"), dpi=300); plt.close(fig)
print("figures:", sorted(os.listdir(FIG)))

          task kind model  units  workers   wall_s  worker_cpu_s  wall_s_per_unit  single_core_s_per_row machine
      SHAP_GNB SHAP   GNB    530        1   28.332        27.484            0.053                    NaN      pc
       SHAP_LR SHAP    LR   2029        1    0.061         0.061            0.000                    NaN      pc
       SHAP_DT SHAP    DT   2029        6    6.729         6.062            0.003                  0.003      pc
      SHAP_SVC SHAP   SVC    530        1   56.980        56.656            0.108                    NaN      pc
      SHAP_MLP SHAP   MLP    530        1   28.414        33.812            0.054                    NaN      pc
     SHAP_LGBM SHAP  LGBM   2029        6  103.894       600.562            0.051                  0.296      pc
      SHAP_CAT SHAP   CAT   2029        6  121.305       693.391            0.060                  0.342      pc
      SHAP_XGB SHAP   XGB   2029        6  141.238       821.047            0.070               

In [11]:
# Cell 11 - report and environment log
def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


env = tc.environment(MACHINE)
write_json({**env, "stage": STAGE, "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "pins_differ": {k: list(v) for k, v in tc.check_pins(PROJECT_DIR).items()}, "test_mode": TEST_MODE,
            "freeze_sha256": FREEZE_SHA256, "test_predictions_sha256": PRED_SHA256, "tree_workers": TREE_WORKERS,
            "seeds": {"shap_permutation": SEED_SHAP_PERM, "perm_seeds": PERM_SEEDS, "lime": LIME_SEEDS, "cases": SEED_CASES},
            "case_panel_sha256": tc.sha256_file(panel_path)}, os.path.join(OUT, f"env_log_stage6_{MACHINE}.json"))
ag = GPAIRS
L = ["# Stage 6 report: SHAP and LIME explanations", "",
     ("**TEST MODE (row limits set): not thesis results.**\n" if TEST_MODE else "")
     + f"Generated by `{STAGE}` (config {tc.CONFIG_VERSION}) on {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC, machine {MACHINE}, "
     f"Python {env['python']}, shap {shap.__version__}. Freeze {FREEZE_SHA256[:16]}; predictions {PRED_SHA256[:16]}.", "",
     f"Samples: background {len(BG)} development rows (tree, linear), {len(BG100)} (permutation); global sample {len(GLOBAL)} test rows "
     f"(tree, linear); core {len(CORE)} (permutation, and all cross-model comparisons); panel {len(PANEL)} cases "
     f"({', '.join(f'{c} {PANEL_PER_CAT}' for c in CATEGORIES)}).", "",
     "## SHAP additivity (fidelity)", "", "| Model | Seed | Explainer | Scale | Rows | Max abs error | Mean abs error |", "| --- | --- | --- | --- | --- | --- | --- |"]
L += [f"| {r.model} | {r.seed} | {r.explainer} | {r.output_scale} | {r.n_rows} | {r.max_abs_additivity_error:.2e} | {r.mean_abs_additivity_error:.2e} |" for r in SFID.itertuples()]
L += ["", f"## Global agreement (core, {len(CORE_USED)} rows; 45 model pairs)", "",
      f"Kendall tau-b: median {ag.kendall_tau_b.median():.3f}, range {ag.kendall_tau_b.min():.3f} to {ag.kendall_tau_b.max():.3f}. "
      f"Top-5 Jaccard: median {ag.top5_jaccard.median():.3f}, range {ag.top5_jaccard.min():.3f} to {ag.top5_jaccard.max():.3f}.", "",
      "Top-5 predictors per model (core): " + "; ".join(f"{m}: " + ", ".join(np.array(FEATURES)[np.argsort(-IMP_CORE[m], kind='stable')[:5]]) for m in MODELS), "",
      "## Stability (5 frozen bootstrap resamples; own sample)", "", "| Model | tau-b mean | tau-b min | top-5 Jaccard mean | top-5 Jaccard min |", "| --- | --- | --- | --- | --- |"]
gs = GSTAB[GSTAB["sample"] == "own"].groupby("model")
L += [f"| {m} | {gs.kendall_tau_b.mean()[m]:.3f} | {gs.kendall_tau_b.min()[m]:.3f} | {gs.top5_jaccard.mean()[m]:.3f} | {gs.top5_jaccard.min()[m]:.3f} |" for m in MODELS if m in gs.groups]
L += ["", "## Permutation-explainer seed check (core)", "", "| Model | Rows | Global tau-b | Global top-5 Jaccard | Mean abs diff / mean abs SHAP | Pearson (all values) |",
      "| --- | --- | --- | --- | --- | --- |"]
L += [f"| {r.model} | {r.n_rows} | {r.kendall_tau_b_global:.3f} | {r.top5_jaccard_global:.3f} | {r.mean_abs_diff_over_mean_abs_shap:.4f} | {r.pearson_all_values:.4f} |"
      for r in SEEDCHK.itertuples()]
L += ["", "## Local (30 cases): medians by model", "", "| Model | SHAP-LIME tau-b | SHAP-LIME top-5 Jaccard | Sign agreement (SHAP top-5) | LIME R2 | LIME gap | LIME seed tau-b | LIME seed top-5 Jaccard |",
      "| --- | --- | --- | --- | --- | --- | --- | --- |"]
L += [f"| {m} | {r.kendall_tau_b_abs:.3f} | {r.top5_jaccard:.3f} | {r.sign_agreement_shap_top5:.3f} | {r.lime_r2_mean:.3f} | {r.lime_abs_gap_mean:.3f} | "
      f"{r.lime_seed_kendall_tau_b_abs:.3f} | {r.lime_seed_top5_jaccard:.3f} |" for m, r in LSUM.iterrows()]
L += ["", "## Operational time (this machine)", "", "| Task | Units | Workers | Wall s | Wall s per unit |", "| --- | --- | --- | --- | --- |"]
L += [f"| {r.task} | {r.units} | {r.workers} | {r.wall_s:.1f} | {r.wall_s_per_unit:.4f} |" for r in TIM.itertuples()]
L += ["", "Explanations describe fitted model behaviour on self-reported survey answers; they are not causal or diagnostic (Ch3 3.5). "
      "Agreement measures consistency between methods or models, not correctness. Table 3.2 timings come from the Stage 7 benchmark."]
with open(os.path.join(OUT, "stage6_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(L) + "\n")
print("\n".join(L))
print("\nSend back: everything in stage6_outputs/ except the _work/ folder (shap/, lime/ and figures/ included).")

# Stage 6 report: SHAP and LIME explanations

Generated by `stage6_explanations.ipynb` (config v1.0) on 2026-10-15 21:22 UTC, machine pc, Python 3.13.0, shap 0.52.0. Freeze 1dd5bf06708d1483; predictions b9813461cd03e455.

Samples: background 500 development rows (tree, linear), 100 (permutation); global sample 2000 test rows (tree, linear); core 500 (permutation, and all cross-model comparisons); panel 30 cases (high_probability 5, borderline 5, TP 5, FN 5, FP 5, TN 5).

## SHAP additivity (fidelity)

| Model | Seed | Explainer | Scale | Rows | Max abs error | Mean abs error |
| --- | --- | --- | --- | --- | --- | --- |
| LR | 1 | LinearExplainer (500 bg) | log-odds | 2029 | 2.22e-15 | 2.24e-16 |
| DT | 1 | TreeExplainer (interventional, 500 bg) | probability | 2029 | 6.42e-08 | 1.15e-08 |
| RF | 1 | TreeExplainer (interventional, 500 bg) | probability | 2029 | 1.67e-08 | 3.40e-09 |
| SVC | 1 | PermutationExplainer (100 bg, max_evals 250) | probability | 530 | 1.78e-15 | 1.70e-16 |
| S